In [3]:
# import necessary packages
import pandas as pd
import numpy as np
import xarray as xr

In [4]:
# import google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# load seasonal_avg pivoted dataframe
eea_seasonal_avg = pd.read_csv('/content/drive/My Drive/capstone_data/csv/seasonal/seasonal_average/eastern_east_africa_CanESM5_merged_seasonal_avg_pivoted.csv')

In [10]:
# take the spatial mean
eea_seasonal_avg = (eea_seasonal_avg  # Use the ensemble_means DataFrame
                              .groupby(['season', 'year', 'lead_category'])[['predicted_precip', 'precip']]
                              .mean().reset_index())

In [16]:
# Compute the Tercile Cutoffs for each unique season and lead category
def get_tercile_cutoffs(df):
    return df.quantile([0.33, 0.66], axis=1)

# Assign the Tercile Category for both predicted_precip and precip
def assign_tercile_category(value, lower_cutoff, upper_cutoff):
    if value <= lower_cutoff:
        return 'Low'
    elif value <= upper_cutoff:
        return 'Medium'
    else:
        return 'High'

In [11]:
eea_seasonal_avg

,season,year,lead_category,predicted_precip,precip
0,MAM,1991,medium,2.509296,72.233860
1,MAM,1991,short,2.738421,72.233860
2,MAM,1992,long,2.381945,51.666221
3,MAM,1992,medium,2.581496,51.666221
4,MAM,1992,short,2.865272,51.666221
...,...,...,...,...,...
178,OND,2020,long,1.872840,39.041262
179,OND,2020,medium,1.824814,39.041262
180,OND,2020,short,1.688120,39.041262
181,OND,2024,medium,1.927668,48.014006


In [13]:
precip_terciles = eea_seasonal_avg.groupby('season')['precip'].quantile([1/3, 2/3]).unstack()
precip_terciles.columns = ['lower_tercile', 'upper_tercile']

predicted_precip_terciles = eea_seasonal_avg.groupby('season')['predicted_precip'].quantile([1/3, 2/3]).unstack()
predicted_precip_terciles.columns = ['lower_tercile', 'upper_tercile']

print("Predicted Precip terciles:")
print(predicted_precip_terciles)

print("Precip terciles:")
print(precip_terciles)

Predicted Precip terciles:
        lower_tercile  upper_tercile
season                              
MAM          2.225743       2.374293
OND          1.810894       2.042225
Precip terciles:
        lower_tercile  upper_tercile
season                              
MAM         59.776036      72.233860
OND         45.307526      55.295319


In [15]:
# subset to 1993 to 2024
current_df = eea_seasonal_avg.query('year >= 1993 and year <= 2024')

current_df['model'] = 'CanESM5'
current_df['region'] = 'eastern_east_africa'

model = 'CanESM5'
region = 'eastern_east_africa'

# Initialize the `agreement` column
current_df['agreement'] = None

# Iterate over each unique combination of month and lead_time, over all years
for (season, lead_category), group in current_df.groupby(['season', 'lead_category']):
    # Compute tercile cutoffs for predicted_precip and precip for this group (across all years)
    cutoffs = get_tercile_cutoffs(group[['predicted_precip', 'precip']])

    # Extract cutoffs for predicted_precip and precip separately
    lower_cutoff_predicted = cutoffs.loc[0.33, 'predicted_precip']
    upper_cutoff_predicted = cutoffs.loc[0.66, 'predicted_precip']

    lower_cutoff_precip = cutoffs.loc[0.33, 'precip']
    upper_cutoff_precip = cutoffs.loc[0.66, 'precip']

    # assign tercile categories for predicted_precip and precip
    group[f'{model}_tercile_class'] = group['predicted_precip'].apply(assign_tercile_category, args=(lower_cutoff_predicted, upper_cutoff_predicted))
    group['chirps_tercile_class'] = group['precip'].apply(assign_tercile_category, args=(lower_cutoff_precip, upper_cutoff_precip))

    # Calculate agreement: 1 if terciles agree, 0 otherwise
    group['agreement'] = (group[f'{model}_tercile_class'] == group['chirps_tercile_class']).astype(int)

    # Update the DataFrame with the new columns
    current_df.loc[group.index, f'{model}_tercile_class'] = group[f'{model}_tercile_class']
    current_df.loc[group.index, 'chirps_tercile_class'] = group['chirps_tercile_class']
    current_df.loc[group.index, 'agreement'] = group['agreement']
    current_df.loc[group.index, 'model'] = model
    current_df.loc[group.index, 'region'] = region

# Select the desired columns for the final DataFrame
final_columns = ['year', 'lead_time', 'month', 'predicted_precip', 'precip', f'{model}_tercile_class', 'chirps_tercile_class', 'agreement', 'model', 'region']
final_df = current_df[final_columns]


<ipython-input-15-0c5c77530293>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_df['model'] = 'CanESM5'
<ipython-input-15-0c5c77530293>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_df['region'] = 'eastern_east_africa'
<ipython-input-15-0c5c77530293>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_

KeyError: 'predicted_precip'